# Home Loan Default Prediction — Deep Learning with Keras & TensorFlow

**Course-End Project — Deep Learning with Keras and TensorFlow**

**Problem Statement:** For a safe and secure lending experience, it's important to analyze past
data. In this project we build a deep learning model to predict the chance of default for future
loans using historical data. The dataset is highly imbalanced and includes many features, which
makes the problem more challenging.

**Objective:** Create a model that predicts whether an applicant will be able to repay a loan,
using historical data.

**Domain:** Finance

**Steps covered in this notebook:**
1. Load the dataset
2. Check for null values
3. Print the percentage of default vs payer for the `TARGET` column
4. Balance the dataset
5. Plot the balanced / imbalanced data
6. Encode the columns required for the model
7. Calculate sensitivity (recall) as a metric
8. Calculate the area under the ROC curve (AUC-ROC)

In [ ]:
%pip install tensorflow imbalanced-learn -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
)

from imblearn.over_sampling import RandomOverSampler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.metrics import Recall
from tensorflow.keras.callbacks import EarlyStopping

print("TensorFlow version:", tf.__version__)

## 1. Load the Dataset

In [ ]:
# NOTE: keep the CSV in the same folder as this notebook (rename if needed)
# so the path below matches the file you upload to GitHub.
df = pd.read_csv("loan_data.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

## 2. Check for Null Values

In [ ]:
null_counts = df.isnull().sum()
null_counts[null_counts > 0].sort_values(ascending=False)

In [ ]:
null_pct = df.isnull().mean() * 100
null_pct[null_pct > 0].sort_values(ascending=False)

## 3. Percentage of Default vs Payer (`TARGET` column)

In [ ]:
target_counts = df["TARGET"].value_counts()
target_pct = df["TARGET"].value_counts(normalize=True) * 100

print("Counts:\n", target_counts)
print("\nPercentage:\n", target_pct.round(2))

In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(x="TARGET", data=df)
plt.title("Loan Default Distribution (Imbalanced)")
plt.xlabel("Target (0 = Repaid, 1 = Default)")
plt.ylabel("Number of Applicants")
plt.show()

## 4. Preprocessing

Drop the ID column, split into train/test, then impute missing values and encode
categorical columns.

In [ ]:
df_model = df.drop("SK_ID_CURR", axis=1)

X = df_model.drop("TARGET", axis=1)
y = df_model["TARGET"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(y_train.value_counts())

In [ ]:
# Identify numerical and categorical columns
numerical_cols = X_train.select_dtypes(exclude=["object"]).columns
categorical_cols = X_train.select_dtypes(include=["object"]).columns

print("Numerical columns:", len(numerical_cols))
print("Categorical columns:", len(categorical_cols))

In [ ]:
# Numerical missing values -> median
num_imputer = SimpleImputer(strategy="median")
X_train_num = num_imputer.fit_transform(X_train[numerical_cols])
X_test_num = num_imputer.transform(X_test[numerical_cols])

# Categorical missing values -> most frequent
cat_imputer = SimpleImputer(strategy="most_frequent")
X_train_cat = cat_imputer.fit_transform(X_train[categorical_cols])
X_test_cat = cat_imputer.transform(X_test[categorical_cols])

## 5. Encode Categorical Columns

In [ ]:
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

X_train_encoded = encoder.fit_transform(X_train_cat)
X_test_encoded = encoder.transform(X_test_cat)

X_train_final = np.hstack([X_train_num, X_train_encoded]).astype(np.float32)
X_test_final = np.hstack([X_test_num, X_test_encoded]).astype(np.float32)

print(X_train_final.dtype)
print(X_train_final.shape, X_test_final.shape)

In [ ]:
scaler = StandardScaler()

X_train_final = scaler.fit_transform(X_train_final)
X_test_final = scaler.transform(X_test_final)

## 6. Balance the Training Data

The dataset is highly imbalanced (~92% repaid vs ~8% default). We balance the **training set
only** (never the test set, so evaluation stays realistic) using random oversampling of the
minority class.

In [ ]:
print("Before balancing:", np.bincount(y_train))

ros = RandomOverSampler(random_state=42)
X_train_bal, y_train_bal = ros.fit_resample(X_train_final, y_train)

print("After balancing:", np.bincount(y_train_bal))

In [ ]:
before = pd.Series(y_train).value_counts().sort_index()
after = pd.Series(y_train_bal).value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

sns.barplot(x=before.index, y=before.values, ax=axes[0])
axes[0].set_title("Training Data (Before Balancing)")
axes[0].set_xlabel("Target")
axes[0].set_ylabel("Count")

sns.barplot(x=after.index, y=after.values, ax=axes[1])
axes[1].set_title("Training Data (After Balancing)")
axes[1].set_xlabel("Target")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

## 7. Build the Deep Learning Model

In [ ]:
model = Sequential([
    Input(shape=(X_train_bal.shape[1],)),

    Dense(128, activation="relu"),
    Dropout(0.3),

    Dense(64, activation="relu"),
    Dropout(0.2),

    Dense(32, activation="relu"),

    Dense(1, activation="sigmoid"),
])

model.summary()

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy", Recall(name="sensitivity")],
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
)

In [ ]:
history = model.fit(
    X_train_bal,
    y_train_bal,
    epochs=30,
    batch_size=256,
    validation_split=0.2,
    callbacks=[early_stop],
)

In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="validation")
plt.title("Loss")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["sensitivity"], label="train")
plt.plot(history.history["val_sensitivity"], label="validation")
plt.title("Sensitivity (Recall)")
plt.xlabel("Epoch")
plt.legend()

plt.tight_layout()
plt.show()

## 8. Model Evaluation — Sensitivity & AUC-ROC

Evaluate on the untouched, original (imbalanced) test set.

In [ ]:
y_pred_prob = model.predict(X_test_final).ravel()
y_pred = (y_pred_prob >= 0.5).astype(int)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
sensitivity = recall_score(y_test, y_pred, zero_division=0)  # a.k.a. recall / TPR
f1 = f1_score(y_test, y_pred, zero_division=0)
auc = roc_auc_score(y_test, y_pred_prob)

print(f"Accuracy:    {accuracy:.4f}")
print(f"Precision:   {precision:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"F1 score:    {f1:.4f}")
print(f"AUC-ROC:     {auc:.4f}")

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(4, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Repaid", "Default"],
            yticklabels=["Repaid", "Default"])
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)

plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Sensitivity)")
plt.title("ROC Curve")
plt.legend()
plt.show()

## Summary

- The dataset is highly imbalanced (~92% repaid vs ~8% default).
- Missing values were imputed (median for numerical, most-frequent for categorical) and
  categorical columns were one-hot encoded.
- The training set was balanced via random oversampling of the minority class before training.
- A feedforward neural network (Keras/TensorFlow) was trained with early stopping on
  validation loss.
- Sensitivity (recall) and AUC-ROC were used as the key evaluation metrics, since accuracy
  alone is misleading on imbalanced data.